# 02 - Preprocessing et Encodage

## Women's E-Commerce Clothing Reviews

Dans ce notebook, je prépare les données nettoyées pour les algorithmes de classification
de Machine Learning classique.

Le premier notebook (`01_eda_and_cleaning.ipynb`) avait pour objectif de comprendre
et nettoyer les données. Ici, je transforme ces données afin qu'elles puissent être
utilisées par les modèles.

Les principales étapes sont :

- charger le dataset nettoyé ;
- séparer les variables explicatives `X` et la cible `y` ;
- supprimer les identifiants non informatifs ;
- séparer les données en train et test ;
- distinguer les variables numériques, catégorielles et textuelles ;
- préparer les variables numériques ;
- encoder les variables catégorielles ;
- transformer les textes avec TF-IDF ;
- construire un préprocesseur complet avec `ColumnTransformer` ;
- apprendre le prétraitement uniquement sur le jeu d'entraînement ;
- transformer les jeux train et test ;
- vérifier le résultat final.

L'objectif est d'obtenir des données numériques prêtes pour le notebook
`03_classification.ipynb`.

Aucun Deep Learning n'est utilisé dans cette étape.


## 1. Importation des bibliothèques

J'importe les bibliothèques nécessaires pour :

- manipuler les données avec pandas ;
- gérer les chemins avec pathlib ;
- séparer les données en train et test ;
- construire des pipelines de prétraitement ;
- imputer les valeurs manquantes ;
- standardiser les variables numériques ;
- encoder les variables catégorielles ;
- transformer les textes avec TF-IDF.


In [ ]:
import warnings

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


ModuleNotFoundError: No module named 'numpy'

## 2. Chargement du dataset nettoyé

Le premier notebook a produit le fichier :

`data/processed/reviews_clean.csv`

Je charge ce fichier plutôt que le dataset brut, car le nettoyage a déjà été effectué
dans le notebook précédent.


In [ ]:
PROJECT_ROOT = Path("..")

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "reviews_clean.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset nettoyé chargé avec succès.")
print(f"Chemin : {DATA_PATH}")
print(f"Dimensions : {df.shape}")


## 3. Première vérification

Je regarde quelques lignes et les informations générales afin de vérifier que le
dataset correspond bien au résultat du notebook de nettoyage.


In [ ]:
display(df.head())

print("\nInformations générales :")
df.info()


## 4. Définition de la cible

La cible du problème de classification est `Recommended IND`.

Elle possède deux valeurs :

- `0` : la cliente ne recommande pas le produit ;
- `1` : la cliente recommande le produit.

Je sépare donc :

- `X` : les variables utilisées pour effectuer la prédiction ;
- `y` : la variable à prédire.


In [ ]:
TARGET_COL = "Recommended IND"

X = df.drop(columns=[TARGET_COL]).copy()
y = df[TARGET_COL].copy()

print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

print("\nDistribution de la cible :")
display(y.value_counts().to_frame("Count"))

print("\nProportion de chaque classe :")
display(
    (y.value_counts(normalize=True) * 100)
    .round(2)
    .to_frame("Percentage")
)


## 5. Vérification de la cible

Avant d'aller plus loin, je vérifie que la cible contient uniquement les deux classes
attendues : `0` et `1`.

Cette vérification permet de s'assurer qu'il s'agit bien d'une classification binaire.


In [ ]:
print("Valeurs présentes dans la cible :")
print(sorted(y.dropna().unique()))

if set(y.dropna().unique()).issubset({0, 1}):
    print("✓ La cible est bien binaire.")
else:
    print("⚠ Vérifier les valeurs présentes dans la cible.")


## 6. Suppression de l'identifiant

`Clothing ID` identifie un produit, mais ce nombre ne représente pas une quantité
numérique au sens classique.

Par exemple, le produit 1000 n'est pas « plus grand » que le produit 500.

Je retire donc `Clothing ID` du jeu de variables explicatives pour éviter de traiter
un identifiant comme une variable quantitative.


In [ ]:
if "Clothing ID" in X.columns:
    X = X.drop(columns=["Clothing ID"])
    print("✓ Clothing ID supprimée.")
else:
    print("✓ Clothing ID n'est pas présente.")

print("\nVariables restantes :")
print(X.columns.tolist())


## 7. Séparation train / test

Je divise les données en deux parties :

- `80 %` pour l'entraînement ;
- `20 %` pour le test.

Le jeu d'entraînement sert à apprendre le modèle et le prétraitement.

Le jeu de test est conservé pour évaluer le modèle sur des données qu'il n'a pas vues
pendant l'apprentissage.

J'utilise `stratify=y` afin de conserver approximativement la même proportion des
classes `0` et `1` dans les deux jeux.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Dimensions après séparation :")
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")


## 8. Vérification de la distribution de la cible après le split

Je vérifie que la proportion des classes est restée proche de celle du dataset initial.

Cette vérification permet de confirmer que `stratify=y` a bien conservé la structure
de la cible dans les deux sous-ensembles.


In [ ]:
distribution_split = pd.DataFrame({
    "Train (%)": y_train.value_counts(normalize=True).sort_index() * 100,
    "Test (%)": y_test.value_counts(normalize=True).sort_index() * 100
}).round(2)

display(distribution_split)


## 9. Identification des familles de variables

Les variables n'ont pas toutes le même type.

Je les sépare en trois familles :

### Variables numériques

- `Age`
- `Rating`
- `Positive Feedback Count`

### Variables catégorielles

- `Division Name`
- `Department Name`
- `Class Name`

### Variables textuelles

- `Title`
- `Review Text`

Cette séparation est nécessaire car chaque famille nécessite une transformation
différente.


In [ ]:
numeric_features = [
    "Age",
    "Rating",
    "Positive Feedback Count"
]

categorical_features = [
    "Division Name",
    "Department Name",
    "Class Name"
]

text_features = [
    "Title",
    "Review Text"
]

print("Variables numériques :", numeric_features)
print("Variables catégorielles :", categorical_features)
print("Variables textuelles :", text_features)


## 10. Vérification des colonnes

Je vérifie que toutes les colonnes nécessaires existent bien dans `X_train`.

Cela évite de construire un pipeline avec un nom de colonne incorrect.


In [ ]:
all_expected_features = (
    numeric_features
    + categorical_features
    + text_features
)

missing_features = [
    col for col in all_expected_features
    if col not in X_train.columns
]

if not missing_features:
    print("✓ Toutes les variables nécessaires sont présentes.")
else:
    print("⚠ Variables manquantes :", missing_features)


## 11. Prétraitement des variables numériques

Les variables numériques sont :

- `Age`
- `Rating`
- `Positive Feedback Count`

Je construis un pipeline avec deux opérations :

1. `SimpleImputer(strategy="median")` :
   remplace une éventuelle valeur numérique manquante par la médiane ;

2. `StandardScaler()` :
   standardise les variables afin de les placer sur des échelles comparables.

Même si le notebook précédent ne laisse plus de valeurs numériques manquantes, garder
l'imputation dans le pipeline rend le traitement plus robuste.


In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

print("✓ Pipeline numérique créé.")
print(numeric_pipeline)


## 12. Prétraitement des variables catégorielles

Les variables catégorielles contiennent des modalités comme :

- `General`
- `Dresses`
- `Tops`
- `Knits`
- etc.

Un modèle classique ne peut pas utiliser directement ces chaînes de caractères.

J'utilise donc :

1. `SimpleImputer(strategy="most_frequent")` pour traiter une éventuelle catégorie
   manquante ;

2. `OneHotEncoder(handle_unknown="ignore")` pour transformer chaque catégorie en
   variables numériques 0/1.

`handle_unknown="ignore"` permet au pipeline de fonctionner même si une catégorie
apparaît dans le test alors qu'elle n'était pas présente dans l'entraînement.


In [ ]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

print("✓ Pipeline catégoriel créé.")
print(categorical_pipeline)


## 13. Prétraitement du texte avec TF-IDF

Les variables `Title` et `Review Text` contiennent du texte libre.

Un algorithme classique ne peut pas utiliser directement une phrase complète.
Je transforme donc le texte en variables numériques avec **TF-IDF**.

TF-IDF attribue un poids aux mots selon leur importance dans les documents.

J'utilise deux vectoriseurs :

- un pour `Title` ;
- un pour `Review Text`.

J'utilise des unigrammes et bigrammes (`ngram_range=(1, 2)`) afin de conserver des mots
seuls et certaines associations de deux mots.

Je limite également le nombre de caractéristiques pour éviter de créer une matrice
inutilement énorme.


In [ ]:
title_tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=3000
)

review_tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=10000
)

print("✓ TF-IDF pour Title créé.")
print("✓ TF-IDF pour Review Text créé.")


## 14. Construction du préprocesseur complet

J'ai maintenant quatre traitements différents :

- variables numériques → imputation + standardisation ;
- variables catégorielles → imputation + One-Hot Encoding ;
- `Title` → TF-IDF ;
- `Review Text` → TF-IDF.

`ColumnTransformer` permet d'appliquer chaque transformation uniquement aux colonnes
qui lui correspondent, puis de combiner automatiquement les résultats.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        ),
        (
            "title_tfidf",
            title_tfidf,
            "Title"
        ),
        (
            "review_tfidf",
            review_tfidf,
            "Review Text"
        )
    ],
    remainder="drop"
)

print("✓ Préprocesseur complet créé.")
print(preprocessor)


## 15. Apprentissage du prétraitement sur le train uniquement

Cette étape est très importante.

J'utilise :

`fit_transform(X_train)`

sur les données d'entraînement.

Le `fit` signifie que le préprocesseur apprend ses paramètres à partir du train :

- médianes ;
- catégories rencontrées ;
- vocabulaire TF-IDF ;
- paramètres de standardisation.

Le test ne doit pas participer à cet apprentissage, afin d'éviter la **fuite de données
(data leakage)**.


In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

print("✓ Prétraitement appris sur le jeu d'entraînement.")
print("Dimensions de X_train après preprocessing :", X_train_processed.shape)


## 16. Transformation du jeu de test

Pour le test, je n'utilise pas `fit`.

J'utilise uniquement :

`transform(X_test)`

Le test est donc transformé avec les paramètres déjà appris sur le train.


In [ ]:
X_test_processed = preprocessor.transform(X_test)

print("✓ Jeu de test transformé.")
print("Dimensions de X_test après preprocessing :", X_test_processed.shape)


## 17. Vérification des dimensions

Avant le prétraitement, nous avions un petit nombre de colonnes.

Après One-Hot Encoding et TF-IDF, le nombre de caractéristiques augmente fortement.

C'est normal :

- les catégories deviennent plusieurs colonnes binaires ;
- les mots du vocabulaire TF-IDF deviennent des caractéristiques numériques.

Les matrices train et test doivent avoir exactement le même nombre de colonnes.


In [ ]:
print("===== VÉRIFICATION DES DIMENSIONS =====")
print(f"X_train original       : {X_train.shape}")
print(f"X_test original        : {X_test.shape}")
print(f"X_train prétraité      : {X_train_processed.shape}")
print(f"X_test prétraité       : {X_test_processed.shape}")

if X_train_processed.shape[1] == X_test_processed.shape[1]:
    print("✓ Train et test ont le même nombre de caractéristiques.")
else:
    print("⚠ Problème : les dimensions train/test diffèrent.")


## 18. Vérification du format des matrices

Avec TF-IDF, la représentation obtenue est généralement une matrice creuse
(*sparse matrix*).

Cela est normal : la plupart des caractéristiques sont égales à zéro pour une
observation donnée.

Nous gardons donc cette représentation sous forme sparse au lieu de la convertir
inutilement en matrice dense.


In [ ]:
print("Type de X_train_processed :", type(X_train_processed))
print("Type de X_test_processed  :", type(X_test_processed))

if hasattr(X_train_processed, "nnz"):
    print("\nNombre de valeurs non nulles dans X_train :", X_train_processed.nnz)
    print(
        "Taux de remplissage :",
        round(
            X_train_processed.nnz
            / (X_train_processed.shape[0] * X_train_processed.shape[1])
            * 100,
            4
        ),
        "%"
    )


## 19. Récupération des noms des caractéristiques

Je récupère les noms des variables produites par le `ColumnTransformer`.

Cela permet de comprendre les caractéristiques finales envoyées au modèle.

Les caractéristiques proviennent des variables numériques, des variables catégorielles
encodées et du vocabulaire TF-IDF.


In [ ]:
feature_names = preprocessor.get_feature_names_out()

print("Nombre total de caractéristiques :", len(feature_names))
print("\nQuelques caractéristiques :")

for i, feature in enumerate(feature_names[:50], start=1):
    print(f"{i}. {feature}")


## 20. Vérification finale du preprocessing

Je réalise une dernière vérification avant de passer au notebook de classification.

Je vérifie :

- que la cible contient toujours 0 et 1 ;
- que train et test ont le même nombre de caractéristiques ;
- que les matrices ne sont pas vides ;
- que le nombre final de caractéristiques correspond aux noms récupérés.


In [ ]:
print("===== VÉRIFICATION FINALE =====")

print("Classes de la cible :", sorted(y.dropna().unique()))
print("Shape X_train_processed :", X_train_processed.shape)
print("Shape X_test_processed  :", X_test_processed.shape)
print("Nombre de feature names :", len(feature_names))

checks = {
    "Cible binaire": set(y.dropna().unique()).issubset({0, 1}),
    "Même nombre de features train/test":
        X_train_processed.shape[1] == X_test_processed.shape[1],
    "Train non vide":
        X_train_processed.shape[0] > 0 and X_train_processed.shape[1] > 0,
    "Test non vide":
        X_test_processed.shape[0] > 0 and X_test_processed.shape[1] > 0,
    "Features cohérentes":
        X_train_processed.shape[1] == len(feature_names)
}

for name, result in checks.items():
    print(f"{'✓' if result else '✗'} {name}")

if all(checks.values()):
    print("\n✓ Le prétraitement est prêt pour la classification.")
else:
    print("\n⚠ Des vérifications doivent être corrigées.")


# 21. Conclusion

Le prétraitement est maintenant terminé.

Nous avons obtenu :

- `X_train_processed` : données d'entraînement transformées ;
- `X_test_processed` : données de test transformées ;
- `y_train` : cible d'entraînement ;
- `y_test` : cible de test.

Les données sont maintenant sous une forme numérique exploitable par les algorithmes
de classification classiques.

## Étape suivante

Dans `03_classification.ipynb`, nous pourrons entraîner et comparer plusieurs modèles
de Machine Learning classique, par exemple :

- Logistic Regression ;
- Decision Tree ;
- Random Forest ;
- SVM ;
- KNN ;
- Naive Bayes.

Aucun Deep Learning n'est utilisé.
